# Lab 1. Why Time Series and Sequence Data Are Different

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-01-why-time-series-and-sequences-lab.ipynb)

This lab accompanies **Chapter 1: Why Time Series and Sequence Data Are Different** in
**Modern Time Series Analysis and Sequence Learning**.

The purpose of this lab is not only to run code. It is designed for **independent study**.
Before each programming task, you will see the mathematical idea, the modeling question, and the interpretation checklist.

## Learning goals

By the end of this lab, you should be able to:

1. Explain the difference between a stochastic process $\{X_t\}$ and one observed realization $\{x_t\}$.
2. Simulate white noise, binary i.i.d. noise, random walks, and trend-seasonal-noise models.
3. Understand why ordinary i.i.d. machine-learning assumptions fail for time-ordered observations.
4. Compute and interpret simple dependence summaries such as lag plots and sample autocorrelation.
5. Convert a time series into a supervised-learning dataset using sliding windows.
6. Build simple forecasting baselines and evaluate them with train/test splits that respect time order.

## Mathematical background

A **time series model** specifies a joint distribution for a sequence of random variables

$$
\{X_t : t \in T\}.
$$

For this course, we usually study **discrete-time, continuous-state** processes, meaning $t=0,1,2,\dots$ and $X_t \in \mathbb{R}$.

A full time-series model would specify probabilities such as

$$
P(X_1 \le x_1,\ldots,X_n \le x_n).
$$

In practice, many useful models are studied through **first- and second-order moments**:

$$
\mu_t = E[X_t],
\qquad
\gamma(s,t) = \operatorname{Cov}(X_s,X_t).
$$

A single observed dataset

$$
x_1,x_2,\ldots,x_n
$$

is one **realization** of the underlying random process. The difficult part of time-series analysis is that we usually observe only one realization, but the observations are dependent and ordered.


## 0. Setup

This lab uses only common Python libraries. In Google Colab, these libraries are already available.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


## 1. Stochastic process versus realization

A stochastic process $\{X_t\}$ is the mathematical object. A realization $\{x_t\}$ is what we observe.

For example, suppose

$$
X_t \sim N(0,1), \qquad t=1,2,\ldots,n,
$$

independently. Then each simulated path is a different realization of the same model.

### What to observe

- The model is the same across all paths.
- The paths are different because random variables generate different outcomes.
- In real data, we usually get only one path, so we must infer the dependence structure from one realization.


In [ ]:
n = 120
num_paths = 5

time = np.arange(n)
paths = rng.normal(loc=0.0, scale=1.0, size=(num_paths, n))

plt.figure()
for i in range(num_paths):
    plt.plot(time, paths[i], linewidth=1, label=f"realization {i+1}")
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Five realizations of the same Gaussian white-noise model")
plt.xlabel("time $t$")
plt.ylabel("$x_t$")
plt.legend(loc="upper right", ncol=2)
plt.show()


### Reflection question

All five curves were generated from the same mathematical model. Why do they look different?

Write your answer in one or two sentences before moving on.


## 2. White noise: the baseline model with no useful past information

A zero-mean white-noise process satisfies

$$
E[X_t]=0,
\qquad
\operatorname{Var}(X_t)=\sigma^2,
\qquad
\operatorname{Cov}(X_s,X_t)=0 \quad \text{for } s\ne t.
$$

White noise may be independent, but the definition only requires no linear correlation across time.

In forecasting, white noise is the simplest benchmark. If a process is truly independent white noise, then the past does not help predict the future:

$$
P(X_t \le x \mid X_{t-1},X_{t-2},\ldots) = P(X_t \le x).
$$


In [ ]:
n = 300
sigma = 1.0
wn = rng.normal(0, sigma, size=n)

wn_series = pd.Series(wn, name="Gaussian white noise")

plt.figure()
plt.plot(wn_series.index, wn_series.values, linewidth=1)
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Gaussian white noise")
plt.xlabel("time $t$")
plt.ylabel("$x_t$")
plt.show()

print(wn_series.describe())


## 3. Binary i.i.d. process

Another simple process is a binary i.i.d. process:

$$
P(X_t=1)=p,
\qquad
P(X_t=-1)=1-p.
$$

When $p=1/2$, the mean is

$$
E[X_t] = 1\cdot \frac12 + (-1)\cdot \frac12 = 0.
$$

This process is useful because it gives a simple model for random steps in a random walk.


In [ ]:
n = 120
p = 0.5
binary = rng.choice([-1, 1], size=n, p=[1-p, p])

plt.figure()
plt.step(np.arange(n), binary, where="mid")
plt.ylim(-1.3, 1.3)
plt.title("Binary i.i.d. process: $P(X_t=1)=P(X_t=-1)=1/2$")
plt.xlabel("time $t$")
plt.ylabel("$x_t$")
plt.show()

print("Sample mean:", binary.mean())
print("Sample variance:", binary.var(ddof=1))


## 4. Random walk: dependence created by accumulation

Let $X_1,X_2,\ldots$ be i.i.d. noise. The random walk is

$$
S_t = \sum_{i=1}^t X_i,
$$

or equivalently

$$
S_t = S_{t-1}+X_t.
$$

This is a central model for cumulative behavior. Even if the increments $X_t$ are independent, the random walk values $S_t$ are strongly dependent.

If $E[X_t]=0$ and $\operatorname{Var}(X_t)=\sigma^2$, then

$$
E[S_t]=0,
\qquad
\operatorname{Var}(S_t)=t\sigma^2.
$$

So the variance grows with time. This is one reason a random walk is not stationary.


In [ ]:
n = 200
num_walks = 6
steps = rng.choice([-1, 1], size=(num_walks, n))
walks = np.cumsum(steps, axis=1)

plt.figure()
for i in range(num_walks):
    plt.plot(np.arange(n), walks[i], linewidth=1, label=f"walk {i+1}")
plt.axhline(0, linestyle="--", linewidth=1)
plt.title("Several random walks from binary increments")
plt.xlabel("time $t$")
plt.ylabel("$S_t$")
plt.legend(loc="upper left", ncol=2)
plt.show()


### Check the theoretical variance growth

We now simulate many random walks and compare empirical variance at time $t$ with the theoretical value $t\sigma^2$.


In [ ]:
n = 200
num_simulations = 5000
steps = rng.normal(0, 1, size=(num_simulations, n))
walks = np.cumsum(steps, axis=1)

# empirical variance across simulated paths at each time point
empirical_var = walks.var(axis=0, ddof=1)
theoretical_var = np.arange(1, n + 1)

plt.figure()
plt.plot(np.arange(1, n + 1), empirical_var, label="empirical variance")
plt.plot(np.arange(1, n + 1), theoretical_var, linestyle="--", label="theoretical variance $t$")
plt.title("Random-walk variance grows over time")
plt.xlabel("time $t$")
plt.ylabel("variance across paths")
plt.legend()
plt.show()


## 5. Difference operator

The first difference of a sequence is

$$
\nabla S_t = S_t-S_{t-1}.
$$

For a random walk,

$$
S_t=S_{t-1}+X_t,
$$

so

$$
\nabla S_t = X_t.
$$

This means the random walk itself is nonstationary, but its increments may be stationary white noise.


In [ ]:
n = 250
steps = rng.normal(0, 1, size=n)
random_walk = np.cumsum(steps)
first_diff = np.diff(random_walk)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=False)
axes[0].plot(random_walk)
axes[0].set_title("Random walk $S_t$")
axes[0].set_xlabel("time $t$")
axes[0].set_ylabel("$S_t$")

axes[1].plot(first_diff)
axes[1].axhline(0, linestyle="--", linewidth=1)
axes[1].set_title("First difference $S_t-S_{t-1}$")
axes[1].set_xlabel("time $t$")
axes[1].set_ylabel("difference")
plt.tight_layout()
plt.show()

print("Variance of first half of random walk:", np.var(random_walk[:n//2], ddof=1))
print("Variance of second half of random walk:", np.var(random_walk[n//2:], ddof=1))
print("Variance of first differences:", np.var(first_diff, ddof=1))


## 6. Trend-seasonal-noise model

A useful first model for observed time series is

$$
X_t = T_t + S_t + E_t,
$$

where:

- $T_t$ is a trend component,
- $S_t$ is a seasonal or periodic component,
- $E_t$ is a noise component.

A simple parametric version is

$$
X_t = \beta_0 + \beta_1 t
+ A\sin\left(\frac{2\pi t}{m}\right)
+ E_t.
$$

Here $m$ is the seasonal period. For example, $m=12$ for monthly data with yearly seasonality, or $m=24$ for hourly data with daily seasonality.


In [ ]:
n = 240
period = 24
t = np.arange(n)

beta0 = 10
beta1 = 0.03
amplitude = 2.0
noise_sd = 0.8

trend = beta0 + beta1 * t
seasonal = amplitude * np.sin(2 * np.pi * t / period)
noise = rng.normal(0, noise_sd, size=n)
y = trend + seasonal + noise

components = pd.DataFrame({
    "trend": trend,
    "seasonal": seasonal,
    "noise": noise,
    "observed": y
})

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)
for ax, col in zip(axes, components.columns):
    ax.plot(t, components[col])
    ax.set_title(col)
    ax.set_ylabel(col)
axes[-1].set_xlabel("time $t$")
plt.tight_layout()
plt.show()


## 7. Sample autocorrelation from scratch

The lag-$h$ sample autocovariance is commonly estimated by

$$
\hat\gamma(h)
= \frac{1}{n}\sum_{t=h+1}^n (x_t-\bar{x})(x_{t-h}-\bar{x}).
$$

The sample autocorrelation is

$$
\hat\rho(h) = \frac{\hat\gamma(h)}{\hat\gamma(0)}.
$$

Autocorrelation measures linear dependence between observations separated by $h$ time steps.


In [ ]:
def sample_autocovariance(x, h):
    """Compute sample autocovariance at lag h using denominator n."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    x_bar = x.mean()
    if h < 0 or h >= n:
        raise ValueError("lag h must satisfy 0 <= h < n")
    return np.sum((x[h:] - x_bar) * (x[:n-h] - x_bar)) / n


def sample_acf(x, max_lag):
    """Compute sample autocorrelation for lags 0,1,...,max_lag."""
    gamma0 = sample_autocovariance(x, 0)
    return np.array([sample_autocovariance(x, h) / gamma0 for h in range(max_lag + 1)])

max_lag = 48
acf_wn = sample_acf(wn, max_lag)
acf_rw = sample_acf(random_walk, max_lag)
acf_seasonal = sample_acf(y, max_lag)

lags = np.arange(max_lag + 1)

plt.figure(figsize=(10, 5))
plt.stem(lags, acf_wn)
plt.axhline(0, linewidth=1)
plt.axhline(2 / np.sqrt(len(wn)), linestyle="--", linewidth=1)
plt.axhline(-2 / np.sqrt(len(wn)), linestyle="--", linewidth=1)
plt.title("Sample ACF of white noise")
plt.xlabel("lag")
plt.ylabel("sample ACF")
plt.show()

plt.figure(figsize=(10, 5))
plt.stem(lags, acf_rw)
plt.axhline(0, linewidth=1)
plt.title("Sample ACF of a random walk")
plt.xlabel("lag")
plt.ylabel("sample ACF")
plt.show()

plt.figure(figsize=(10, 5))
plt.stem(lags, acf_seasonal)
plt.axhline(0, linewidth=1)
plt.title("Sample ACF of trend-seasonal-noise data")
plt.xlabel("lag")
plt.ylabel("sample ACF")
plt.show()


### Interpretation checkpoint

Compare the three ACF plots.

- White noise should have sample autocorrelations close to zero, except for random sampling noise.
- A random walk often has very slowly decaying sample autocorrelation.
- A seasonal series often has peaks near seasonal lags such as $24,48,\ldots$.

These patterns are not proofs, but they are useful diagnostic clues.


## 8. Why random train/test splitting is dangerous for time series

For independent tabular data, random train/test splitting is often reasonable.
For time series, random splitting can leak future information into the training set.

A safer split is chronological:

$$
\text{train} = \{1,\ldots,T\},
\qquad
\text{test} = \{T+1,\ldots,n\}.
$$

In forecasting, the model should only use information available at the forecast origin.


In [ ]:
series = pd.Series(y, index=pd.RangeIndex(n, name="t"), name="y")
train_size = int(0.75 * n)
train = series.iloc[:train_size]
test = series.iloc[train_size:]

plt.figure()
plt.plot(train.index, train.values, label="train")
plt.plot(test.index, test.values, label="test")
plt.axvline(train.index[-1], linestyle="--", linewidth=1)
plt.title("Chronological train/test split")
plt.xlabel("time $t$")
plt.ylabel("$y_t$")
plt.legend()
plt.show()


## 9. Baseline forecasting methods

A forecasting baseline is a simple method that more complicated models must beat.

For a forecast horizon of one step, common baselines include:

1. **Mean forecast**:

$$
\hat{x}_{t+1} = \bar{x}_{1:t}.
$$

2. **Naive forecast**:

$$
\hat{x}_{t+1} = x_t.
$$

3. **Seasonal naive forecast** with period $m$:

$$
\hat{x}_{t+1} = x_{t+1-m}.
$$

For seasonal data, the seasonal naive forecast is often surprisingly strong.


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# One-step-like baselines on the test set.
# At each test time t, we create predictions using only training history and already-observed test past.

history = list(train.values)
mean_preds = []
naive_preds = []
seasonal_naive_preds = []
seasonal_period = 24

for actual in test.values:
    mean_preds.append(np.mean(history))
    naive_preds.append(history[-1])
    if len(history) >= seasonal_period:
        seasonal_naive_preds.append(history[-seasonal_period])
    else:
        seasonal_naive_preds.append(history[-1])
    history.append(actual)

baseline_results = pd.DataFrame({
    "method": ["mean", "naive", "seasonal naive"],
    "MAE": [
        mean_absolute_error(test.values, mean_preds),
        mean_absolute_error(test.values, naive_preds),
        mean_absolute_error(test.values, seasonal_naive_preds),
    ],
    "RMSE": [
        rmse(test.values, mean_preds),
        rmse(test.values, naive_preds),
        rmse(test.values, seasonal_naive_preds),
    ],
})

baseline_results


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(test.index, test.values, label="actual")
plt.plot(test.index, mean_preds, label="mean forecast")
plt.plot(test.index, naive_preds, label="naive forecast")
plt.plot(test.index, seasonal_naive_preds, label="seasonal naive forecast")
plt.title("Baseline forecasts on the test set")
plt.xlabel("time $t$")
plt.ylabel("$y_t$")
plt.legend()
plt.show()


## 10. Turning a time series into supervised-learning data

Many machine-learning models require a matrix of features $X$ and a target vector $y$.
For time series, we can build features from lagged observations.

For a window length $L$, define

$$
\mathbf{x}_t = (x_{t-L+1},x_{t-L+2},\ldots,x_t) \in \mathbb{R}^L
$$

and target

$$
y_t = x_{t+1}.
$$

This converts one sequence into many overlapping training examples.

Important: the row for time $t$ may only use values available at or before time $t$.


In [ ]:
def make_windows(values, window_length, horizon=1):
    """Convert a 1D time series into supervised windows.

    X[i] contains values from time i to i+window_length-1.
    y[i] is the value horizon steps after the end of the input window.
    """
    values = np.asarray(values, dtype=float)
    X = []
    targets = []
    target_times = []
    last_start = len(values) - window_length - horizon + 1
    for start in range(last_start):
        end = start + window_length
        target_time = end + horizon - 1
        X.append(values[start:end])
        targets.append(values[target_time])
        target_times.append(target_time)
    return np.asarray(X), np.asarray(targets), np.asarray(target_times)

L = 24
H = 1
X_all, y_all, target_times = make_windows(series.values, window_length=L, horizon=H)

print("Feature matrix shape:", X_all.shape)
print("Target vector shape:", y_all.shape)
print("First input window:")
print(np.round(X_all[0], 2))
print("First target:", round(y_all[0], 2), "at time", target_times[0])


### Visualize one input window and target


In [ ]:
window_id = 30
input_start = window_id
input_end = window_id + L
forecast_time = input_end + H - 1

plt.figure(figsize=(10, 5))
plt.plot(series.index, series.values, linewidth=1, label="full series")
plt.scatter(np.arange(input_start, input_end), series.values[input_start:input_end], label="input window")
plt.scatter([forecast_time], [series.values[forecast_time]], marker="x", s=100, label="target")
plt.axvline(input_end - 1, linestyle="--", linewidth=1)
plt.title("One supervised-learning example from a time series")
plt.xlabel("time $t$")
plt.ylabel("$y_t$")
plt.legend()
plt.show()


## 11. A first machine-learning forecast: linear regression on lag windows

We now fit a simple linear regression model:

$$
\hat{x}_{t+1}
= b + w_1 x_{t-L+1}+w_2 x_{t-L+2}+\cdots+w_L x_t.
$$

This is not yet an AR model in the strict statistical sense, but it introduces the key supervised-learning idea for sequence data.


In [ ]:
# Chronological split by target time.
train_mask = target_times < train_size
test_mask = target_times >= train_size

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_test, y_test = X_all[test_mask], y_all[test_mask]
times_test = target_times[test_mask]

model = LinearRegression()
model.fit(X_train, y_train)
ml_preds = model.predict(X_test)

ml_mae = mean_absolute_error(y_test, ml_preds)
ml_rmse = rmse(y_test, ml_preds)

print("Linear lag-window model")
print("MAE:", round(ml_mae, 3))
print("RMSE:", round(ml_rmse, 3))


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(times_test, y_test, label="actual")
plt.plot(times_test, ml_preds, label="linear window forecast")
plt.title("Linear regression using lag windows")
plt.xlabel("time $t$")
plt.ylabel("$y_t$")
plt.legend()
plt.show()

pd.DataFrame({
    "method": ["linear lag-window model"],
    "MAE": [ml_mae],
    "RMSE": [ml_rmse]
})


## 12. Exercise: compare window lengths

Run the next cell. It compares several input window lengths.

Before running the cell, predict what might happen:

- A very short window may not capture seasonality.
- A window near the seasonal period may perform well.
- A very long window may overfit or add unnecessary features.


In [ ]:
window_lengths = [3, 6, 12, 24, 36, 48]
rows = []

for L in window_lengths:
    X_all_L, y_all_L, target_times_L = make_windows(series.values, window_length=L, horizon=1)
    train_mask_L = target_times_L < train_size
    test_mask_L = target_times_L >= train_size

    X_train_L = X_all_L[train_mask_L]
    y_train_L = y_all_L[train_mask_L]
    X_test_L = X_all_L[test_mask_L]
    y_test_L = y_all_L[test_mask_L]

    model_L = LinearRegression()
    model_L.fit(X_train_L, y_train_L)
    pred_L = model_L.predict(X_test_L)

    rows.append({
        "window_length": L,
        "MAE": mean_absolute_error(y_test_L, pred_L),
        "RMSE": rmse(y_test_L, pred_L),
        "num_features": L,
        "num_train_windows": len(y_train_L),
    })

window_results = pd.DataFrame(rows)
window_results


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(window_results["window_length"], window_results["RMSE"], marker="o")
plt.title("Test RMSE versus window length")
plt.xlabel("window length $L$")
plt.ylabel("test RMSE")
plt.show()


## 13. AI-assisted study component

Use an AI assistant as a study partner, not as a replacement for your own reasoning.

### Prompt 1: Explain the modeling distinction

> I am studying time series. Explain the difference between a stochastic process and a realization. Use the notation $X_t$ and $x_t$, and give one example from simulated white noise.

### Prompt 2: Diagnose a process from plots

> I simulated a random walk and its first difference. The random walk has slowly changing level and increasing variance, while the first difference fluctuates around zero. Explain why the first difference may be more suitable for stationary modeling.

### Prompt 3: Check for leakage

> I converted a time series into supervised-learning windows. Each row uses the past 24 observations to predict the next observation. What are common ways that future information could accidentally leak into my training data?

### Verification checklist for AI answers

After reading the AI answer, verify:

1. Does it distinguish random variables $X_t$ from observations $x_t$?
2. Does it respect time order?
3. Does it avoid random train/test splitting for forecasting?
4. Does it explain why random walks are nonstationary?
5. Does it connect differencing to increments?


## 14. Independent-study tasks

Complete these tasks after finishing the guided sections.

### Task A: Change the binary probability

Set $p=0.7$ in the binary i.i.d. process. Simulate the process and estimate its mean. Compare with the theoretical mean

$$
E[X_t] = 1\cdot p + (-1)(1-p) = 2p-1.
$$

### Task B: Random walk with drift

Simulate

$$
S_t = S_{t-1}+\mu+W_t.
$$

Try $\mu=0.05$. Plot $S_t$ and $\nabla S_t$.

### Task C: Seasonal period

In the trend-seasonal-noise model, change the period from $24$ to $12$. Plot the time series and its sample ACF. What changes?

### Task D: Forecasting baseline

Compare the seasonal naive forecast with the linear lag-window model for the new period.

### Task E: Short written answer

In 5--8 sentences, explain why sequence data require different validation methods than ordinary i.i.d. tabular data.


## 15. Lab summary

In this lab you learned that:

- A time series is one realization of a stochastic process.
- White noise has no useful linear dependence across time.
- A random walk is created by accumulating noise and is usually nonstationary.
- Differencing can convert a random walk into its stationary increments.
- Trend and seasonality can create strong time dependence.
- Sample ACF is a practical first tool for detecting dependence.
- Forecasting requires chronological validation.
- Sliding windows convert a sequence into supervised-learning examples.

These ideas are the foundation for the rest of the course: stationarity, ARMA/ARIMA, spectral analysis, deep learning, and modern sequence modeling.
